# CTMC ベイズサロゲートモデル検証ノートブック

このノートでは、`DeepSets_varSets_forDiagnel`（ベイズ版サロゲート）に対して

- 検証データ上での **Negative Log-Likelihood (NLL)**
- 事後平均による **RMSE**
- **95% credible interval coverage**
- 任意 1 サンプルに対する **posterior shape 可視化**

を一通り行います。

> ⚠️ 注意  
> - このノートブックは、`model_bayes_1122.py` が同じディレクトリにある前提です。  
> - 検証データのロード部分 (`TODO` コメント箇所) は、あなたの環境に合わせて書き換えてください。


In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from tqdm import tqdm

# ローカルモジュールのインポート
# 同じフォルダに model_bayes_1122.py を置いておくこと
from model_bayes_1122 import (
    DeepSets_varSets_forDiagnel,
    varSets_Datasets,
    collate_fn,
    set_device,
)

# ===== パス設定（あなたの環境に合わせて書き換え）=====
DATA_DIR = Path("./data")          # 検証用データが入っているディレクトリ
MODEL_CKPT = Path("./model_bayes_ckpt.pth")  # 学習済みモデルの重み

device = set_device()
print(f"Using device: {device}")

## 検証データの読み込み

ここはあなたの既存の MLE 検証 notebook と同様の形式に合わせてください。
典型的には、以下のように `np.load` や `torch.load` で `states`, `del_t`, `outputs` を取得し、`varSets_Datasets` に渡します。

In [ ]:
# ===== 検証データ読み込み（TODO: あなたの形式に合わせて編集）=====

# 例: npz からロードする場合
# val_npz = np.load(DATA_DIR / "ctmc_val_dataset.npz", allow_pickle=True)
# states_val = val_npz["states"]
# del_t_val = val_npz["del_t"]
# outputs_val = val_npz["outputs"]   # shape (N,3)

# --- ダミーの構造だけ書いておく ---
# 実際には上記例のように読み込み処理を書き換えてください
states_val = np.load(DATA_DIR / "states_val.npy", allow_pickle=True)
del_t_val = np.load(DATA_DIR / "del_t_val.npy", allow_pickle=True)
outputs_val = np.load(DATA_DIR / "outputs_val.npy", allow_pickle=True)

print("#val =", len(states_val))

# Dataset / DataLoader
val_dataset = varSets_Datasets(states_val, del_t_val, outputs_val)
val_loader = DataLoader(
    val_dataset,
    batch_size=1,          # posterior の可視化をしやすいように 1 にしておく
    shuffle=False,
    collate_fn=collate_fn,
)

## 学習済みモデルの読み込み

従来の MLE サロゲートと同様に、学習済みの重みをロードします。

In [ ]:
# ===== モデル定義 & 重みロード =====

# mdn_components は学習時と同じ値にすること（1 or それ以上）
model = DeepSets_varSets_forDiagnel(
    num_categories=4,
    embedding_dim=16,
    token_hidden1=256,
    token_hidden2=512,
    output_hidden1=128,
    output_hidden2=64,
    dropout=0.2,
    input_is_one_based=True,
    device=device,
    mdn_components=1,   # ベイズ版で K=1 から試す場合
)

if MODEL_CKPT.exists():
    state_dict = torch.load(MODEL_CKPT, map_location=device)
    model.load_state_dict(state_dict)
    print(f"Loaded checkpoint from {MODEL_CKPT}")
else:
    print("[Warning] MODEL_CKPT が見つかりません。学習済みの重みを指定してください。")

## 評価用ヘルパー関数

- Negative Log-Likelihood (NLL)
- 事後平均による RMSE
- 95% credible interval coverage

In [ ]:
def evaluate_nll(model, val_loader):
    """平均 Negative Log-Likelihood を計算"""
    model.eval()
    total_logp = 0.0
    count = 0

    with torch.no_grad():
        for state, delta_t, q_true, lengths in tqdm(val_loader, desc="Eval NLL"):
            logp = model.log_prob(state, delta_t, lengths, q_true)  # (B,)
            total_logp += logp.sum().item()
            count += state.size(0)

    avg_nll = - total_logp / max(count, 1)
    return avg_nll


def evaluate_rmse(model, val_loader):
    """事後平均（forward 出力）と真値の RMSE"""
    model.eval()
    total_rmse = 0.0
    count = 0

    with torch.no_grad():
        for state, delta_t, q_true, lengths in tqdm(val_loader, desc="Eval RMSE"):
            state = state.to(model.device)
            delta_t = delta_t.to(model.device)
            q_true = q_true.to(model.device)
            lengths = lengths.to(model.device)

            q_pred = model(state, delta_t, lengths)  # (B,3)
            mse = ((q_pred - q_true)**2).sum(dim=1).sqrt()  # L2 norm per sample
            total_rmse += mse.sum().item()
            count += state.size(0)

    return total_rmse / max(count, 1)

In [ ]:
def evaluate_credible_interval(
    model,
    val_loader,
    num_samples: int = 2000,
    cred: float = 0.95,
):
    """95% credible interval coverage を評価"""
    model.eval()
    device = model.device
    alpha = (1 - cred) / 2.0   # 0.025 (95% CI)

    total = 0
    covered_all_dims = 0
    covered_any_dim = 0

    for state, delta_t, q_true, lengths in tqdm(val_loader, desc=f"Eval {int(cred*100)}% CI"):
        state = state.to(device)
        delta_t = delta_t.to(device)
        q_true = q_true.to(device)
        lengths = lengths.to(device)

        # posterior sample: (num_samples, B, 3)
        samples = model.sample_posterior(
            state, delta_t, lengths, num_samples=num_samples
        )[:, 0, :]   # (S,3)  B=1 前提

        # 各次元ごとに CI を計算
        q_low = torch.quantile(samples, alpha, dim=0)      # (3,)
        q_high = torch.quantile(samples, 1-alpha, dim=0)   # (3,)

        inside = torch.logical_and(q_true[0] >= q_low, q_true[0] <= q_high)  # (3,)

        if inside.all():
            covered_all_dims += 1
        if inside.any():
            covered_any_dim += 1

        total += 1

    coverage_all = covered_all_dims / max(total, 1)
    coverage_any = covered_any_dim / max(total, 1)
    print(f"{int(cred*100)}% CI coverage (all 3 dims inside): {coverage_all:.4f}")
    print(f"{int(cred*100)}% CI coverage (any dim inside)  : {coverage_any:.4f}")
    return coverage_all, coverage_any

## Posterior shape の可視化（1サンプル）

検証データから 1 サンプル選び、その事後分布 p(q | X) をサンプルしてペアプロットします。

In [ ]:
def visualize_posterior_shape(
    model,
    sample_state,
    sample_delta_t,
    sample_lengths,
    q_true,
    num_samples=2000,
    figsize=(10, 10)
):
    model.eval()
    device = model.device

    # to device
    sample_state = sample_state.to(device)
    sample_delta_t = sample_delta_t.to(device)
    sample_lengths = sample_lengths.to(device)
    q_true = q_true.to(device)

    # posterior samples
    with torch.no_grad():
        samples = model.sample_posterior(
            sample_state,
            sample_delta_t,
            sample_lengths,
            num_samples=num_samples,
        )[:, 0, :]  # (S, 3)

    df = pd.DataFrame(
        samples.cpu().numpy(),
        columns=["q12", "q23", "q34"],
    )

    sns.set(style="white", context="talk")
    g = sns.pairplot(
        df,
        diag_kind="kde",
        plot_kws={"alpha": 0.3, "s": 12},
        corner=True,
    )

    # 真値を上書き表示
    true_vals = q_true[0].detach().cpu().numpy()
    dims = ["q12", "q23", "q34"]
    dim_idx = {name: i for i, name in enumerate(dims)}

    for i, d1 in enumerate(dims):
        for j, d2 in enumerate(dims):
            if j >= i:
                continue
            ax = g.axes[i][j]
            ax.scatter(
                true_vals[dim_idx[d2]],
                true_vals[dim_idx[d1]],
                color="red",
                s=80,
                marker="x",
                label="true" if (i == 1 and j == 0) else "",
            )
            if i == 1 and j == 0:
                ax.legend()

    plt.suptitle(
        f"Posterior shape (num_samples={num_samples})\nTrue q = {true_vals.tolist()}",
        y=1.02,
        fontsize=16,
    )
    plt.show()

## 一括評価の実行

- NLL
- RMSE
- 95% CI coverage
を計算します。

In [ ]:
nll = evaluate_nll(model, val_loader)
rmse = evaluate_rmse(model, val_loader)
cov_all, cov_any = evaluate_credible_interval(model, val_loader, num_samples=2000, cred=0.95)

print("\n=== Validation Summary ===")
print(f"NLL  : {nll:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"95% CI coverage (all dims): {cov_all:.4f}")
print(f"95% CI coverage (any dim) : {cov_any:.4f}")

## 任意 1 サンプルの posterior 可視化

検証データの中から任意のインデックスのサンプルを選び、posterior の形状を可視化します。

In [ ]:
# 可視化したいサンプルのインデックス
sample_idx = 0

# DataLoader の順番を固定している前提（shuffle=False）
state_all, delta_t_all, q_true_all, lengths_all = next(iter(DataLoader(
    val_dataset,
    batch_size=len(val_dataset),
    shuffle=False,
    collate_fn=collate_fn,
)))

sample_state = state_all[sample_idx:sample_idx+1]
sample_delta_t = delta_t_all[sample_idx:sample_idx+1]
sample_q_true = q_true_all[sample_idx:sample_idx+1]
sample_lengths = lengths_all[sample_idx:sample_idx+1]

print("True q:", sample_q_true)

visualize_posterior_shape(
    model,
    sample_state,
    sample_delta_t,
    sample_lengths,
    sample_q_true,
    num_samples=3000,
)